In [1]:
# imports
from torchvision import datasets, transforms
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import pandas as pd
import matplotlib.pyplot as plt
import random
import shutil
import re
from collections import Counter, defaultdict
from pathlib import Path
from ultralytics import YOLO

import cv2

%matplotlib inline

# make dirs where needed
Path("dataset").mkdir(exist_ok=True)
Path("dataset/train").mkdir(exist_ok=True)
Path("dataset/test").mkdir(exist_ok=True)
Path("dataset/val").mkdir(exist_ok=True)

In [2]:
## Parameters
# data cleaning
ignored_videos = [
    "V_AIRPLANE_001",
    "V_AIRPLANE_004",
    "V_AIRPLANE_005",
    "V_AIRPLANE_006",
    "V_AIRPLANE_007",
    "V_AIRPLANE_009",
    "V_AIRPLANE_011",
    "V_AIRPLANE_014",
    "V_AIRPLANE_015",
    "V_AIRPLANE_016",
    "V_AIRPLANE_019",
    "V_AIRPLANE_023",
    "V_AIRPLANE_024",
    "V_AIRPLANE_025",
    "V_AIRPLANE_026",
    "V_AIRPLANE_027",
    "V_AIRPLANE_028",
    "V_AIRPLANE_029",
    "V_AIRPLANE_030",
    "V_AIRPLANE_031",
    "V_AIRPLANE_032",
    "V_AIRPLANE_034",
    "V_AIRPLANE_051",
    "V_AIRPLANE_053",
    "V_AIRPLANE_054",
    "V_AIRPLANE_055",
    "V_AIRPLANE_056",
    "V_AIRPLANE_057",
    "V_AIRPLANE_058",
    "V_AIRPLANE_059",
    "V_BIRD_001",
    "V_BIRD_006",
    "V_BIRD_009",
    "V_BIRD_027",
    "V_BIRD_028",
    "V_BIRD_029",
    "V_BIRD_042",
    "V_BIRD_043",
    "V_BIRD_044",
    "V_BIRD_049",
    "V_BIRD_050",
    "V_DRONE_010",
    "V_HELICOPTER_010",
    "V_HELICOPTER_015",
    "V_HELICOPTER_016",
    "V_HELICOPTER_057",
    "V_HELICOPTER_058",
    "V_HELICOPTER_059",
    "V_HELICOPTER_060",
]
# notoriously bad examples:V_AIRPLANE_005,  V_HELICOPTER_057-60 

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

# training split parameters
train_p = 0.7
test_p = 0.15
val_p = 1 - (train_p + test_p) # 0.15

# file paths
dataFilePath = 'unfiltered_data/all_labels.csv'
videoFolderPath = 'unfiltered_data/Data/Video_V'
outFolderName = 'dataset'
outputFolderPath = Path(outFolderName)

classes = {'AIRPLANE', 'BIRD', 'DRONE', 'HELICOPTER'}

# number of epochs to run the CNN on
NUM_EPOCHS = 8


YOLO_EPOCHS = 18
YOLO_STEPS = 5
YOLO_SEED = 15
YOLO_NCHECK = 30

cuda


In [ ]:
# small CNN for the project
# used as a sort of starting point which we'll compare to YOLO results.

# data split
random.seed(YOLO_SEED)

# load data CSV into pandas
df = pd.read_csv(dataFilePath)
df["video"] = df["video"].fillna("")
df["class"] = df["class"].str.strip()

# drop ignored items
dropped = df['video'].str.removesuffix('.mp4').isin(ignored_videos)

df = df[~dropped].reset_index(drop=True)

# index every video file
videos = {v for v in Path(videoFolderPath).rglob("*.mp4") if v.stem not in ignored_videos}

# extract every 5th frame from videos

for v in videos:
    openV = cv2.VideoCapture(str(v))

    # get class if present in string
    classV = next((c for c in classes if c in v.stem.upper()), None)
    vID = v.stem.split('_')[2]

    if not openV.isOpened():
        print('Error opening ' + str(v))
        continue
    else:
        print('opened ' + str(v))
    
    # split sets into train, test, val
    n = random.randint(1, 100)
    plitDir = None
    if n > 100 - (test_p * 100):
        splitDir = 'test'
    elif n > 100 - (test_p* 100) - (val_p* 100):
        splitDir = 'val'
    else:
        splitDir = 'train'
    
    frameNum = 0
    while True:
        s, f = openV.read()

        if not s:
            break
        if frameNum % 5 == 0:
            frame_name = Path(outputFolderPath) / splitDir / f'f_{classV}_{vID}_{frameNum:05d}.jpg'
            if not cv2.imwrite(str(frame_name), f):
                print('Failed to write ' + str(frame_name))
        frameNum += 1
    openV.release()    

In [ ]:
# move frames into class subfolders so ImageFolder can find them: dataset/{split}/{class}/
ignored = {v.split('_', 1)[1] for v in ignored_videos}  # 'V_AIRPLANE_001' -> 'AIRPLANE_001'

for split in ['train', 'val', 'test']:
    for f in (outputFolderPath / split).glob('f_*.jpg'):
        _, classF, vID, _ = f.stem.split('_')
        # drop frames from ignored videos that slipped through extraction
        if f'{classF}_{vID}' in ignored:
            f.unlink()
            continue
        dest = outputFolderPath / split / classF
        dest.mkdir(exist_ok=True)
        shutil.move(str(f), dest / f.name)

for split in ['train', 'val', 'test']:
    print(split, {d.name: len(list(d.glob('*.jpg'))) for d in (outputFolderPath / split).iterdir()})

train {'AIRPLANE': 1395, 'BIRD': 1667, 'DRONE': 5317, 'HELICOPTER': 2323}
val {'AIRPLANE': 319, 'BIRD': 379, 'DRONE': 878, 'HELICOPTER': 563}
test {'AIRPLANE': 124, 'BIRD': 511, 'DRONE': 894, 'HELICOPTER': 503}


In [8]:
# CNN code
# resize images to 128x128 for faster compute, also no need for large image sizes
IMGSize = 128

# train transforms
train_tf = transforms.Compose([
    transforms.Resize((IMGSize, IMGSize)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2),
    transforms.ToTensor()
])
# evaluate transforms
eval_tf = transforms.Compose([transforms.Resize((IMGSize, IMGSize)), transforms.ToTensor()])

train_ds = datasets.ImageFolder(outputFolderPath / 'train', train_tf)
val_ds = datasets.ImageFolder(outputFolderPath / 'val', eval_tf)
test_ds = datasets.ImageFolder(outputFolderPath / 'test', eval_tf)
print(train_ds.class_to_idx, len(train_ds), len(val_ds), len(test_ds))

# batch loaders
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0)
val_dl = DataLoader(val_ds, batch_size=64, num_workers=0)
test_dl = DataLoader(test_ds, batch_size=64, num_workers=0)

{'AIRPLANE': 0, 'BIRD': 1, 'DRONE': 2, 'HELICOPTER': 3} 10702 2139 2032


In [11]:
# function to halve image size
def block(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))

# the model itself
model = nn.Sequential(
    block(3, 32),
    block(32, 64),
    block(64, 128),
    block(128, 256),
    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(),
    nn.Dropout(0.3),
    nn.Linear(256, len(train_ds.classes))
)

model = model.to(device)

# optimizer and loss functions, uses the same optimizer as YOLO
optimiz = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()

# function to run epochs on CNN. returns loss, accuracy
def epoch(dl, train):
    model.train(train)
    tot, corr, loss_sum = 0, 0, 0.0
    with torch.set_grad_enabled(train):
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = loss_fn(out, y)
            if train:
                optimiz.zero_grad()
                loss.backward()
                optimiz.step()
            loss_sum += loss.item() * len(y)
            corr += (out.argmax(1) == y).sum().item()
            tot += len(y)
        return loss_sum / tot, corr / tot

# training loop to run continuously and track best epoch
best = 0
for e in range(NUM_EPOCHS):
    tl, ta = epoch(train_dl, True)
    vl, va = epoch(val_dl, False)
    print(f'epoch {e}: train loss {tl:.3f} acc {ta:.3f} | val loss {vl:.3f} acc {va:.3f}')
    # save model from best epoch
    if va > best:
        best=va
        torch.save(model.state_dict(), 'CNN.pt')

# test best epoch on test set
model.load_state_dict(torch.load('CNN.pt'))
print(epoch(test_dl, False)[1])
    
    

epoch 0: train loss 0.639 acc 0.774 | val loss 0.765 acc 0.733
epoch 1: train loss 0.368 acc 0.878 | val loss 0.728 acc 0.773
epoch 2: train loss 0.279 acc 0.903 | val loss 0.852 acc 0.735
epoch 3: train loss 0.226 acc 0.924 | val loss 0.780 acc 0.755
epoch 4: train loss 0.199 acc 0.931 | val loss 0.789 acc 0.762
epoch 5: train loss 0.179 acc 0.938 | val loss 0.788 acc 0.748
epoch 6: train loss 0.157 acc 0.946 | val loss 0.695 acc 0.751
epoch 7: train loss 0.153 acc 0.947 | val loss 0.762 acc 0.759
0.9271653543307087


The main tradeoff with using a classic CNN model for this project is that we want to accurately track which part of the image is the drone, instead of just classifying the entire image. To do that manually, we would need to first alter the training data to only train on the cropped parts of the images. We then would need (for new data), to accurately draw bounding boxes around objects in real time to then feed into the CNN as it expects objects to be relatively center. There are alternatives to this method however. Yolo algorithms do all of that described with surprising efficiency. We can train a pre-trained YOLO model YOLOv8 via transfer learning to have it learn on our data.

In [12]:
# CNN tester
model.load_state_dict(torch.load('CNN.pt'))
print(epoch(test_dl, False))

(0.2807981448732023, 0.9271653543307087)


In [ ]:
# YOLO code
# reuses the frames already extracted into dataset/{split}/{class}/ for the CNN,
# YOLO looks for a .txt label with the same name next to each image
# written with assistance from Claude Opus 5.5
CLASS_MAP = {c: i for i, c in enumerate(sorted(classes))}

# load CSV into pandas
df = pd.read_csv(dataFilePath)
df["class"] = df["class"].str.strip()

# boxes for each (video, frame), rows in the CSV are 1-indexed frame numbers
boxes = defaultdict(list)
for v, r, c, x, y, w, h in zip(df["video"], df["row"], df["class"], df["x"], df["y"], df["w"], df["h"]):
    boxes[(v.removesuffix(".mp4"), r - 1)].append((CLASS_MAP[c], x, y, w, h))

# write a label file for every frame
for img_path in outputFolderPath.glob("*/*/*.jpg"):
    _, classF, vID, frameNum = img_path.stem.split("_")  # f_DRONE_001_00005
    H, W = cv2.imread(str(img_path)).shape[:2]
    lines = []
    for cls, x, y, w, h in boxes[(f"V_{classF}_{vID}", int(frameNum))]:
        x0, y0 = max(x - 1, 0), max(y - 1, 0)  # MATLAB is 1-indexed
        bw, bh = min(w, W - x0), min(h, H - y0)
        if bw > 1 and bh > 1:
            lines.append(f"{cls} {(x0 + bw / 2) / W:.6f} {(y0 + bh / 2) / H:.6f} {bw / W:.6f} {bh / H:.6f}")
    img_path.with_suffix(".txt").write_text("\n".join(lines))

# yaml for YOLO
yaml = [f"path: {outputFolderPath.resolve()}", "train: train", "val: val", "test: test",
        f"nc: {len(CLASS_MAP)}", "names:"] + [f"  {i}: {c}" for c, i in CLASS_MAP.items()]
(outputFolderPath / "data.yaml").write_text("\n".join(yaml) + "\n")
print(CLASS_MAP)

{'AIRPLANE': 0, 'BIRD': 1, 'DRONE': 2, 'HELICOPTER': 3}


In [ ]:
# training YOLO

yolo = YOLO('yolov8s.pt')
yolo.train(data='dataset/data.yaml', epochs=YOLO_EPOCHS, imgsz=640, batch=16, workers=0)

In [ ]:
# code for producing graphics tied to the training data

bestTrainFolder = 'runs/detect/train-9'
df = pd.read_csv('runs/detect/train-9/results.csv')
df.columns = df.columns.str.strip()

# convergence graph
plt.figure(figsize=(7, 4))
plt.plot(df["epoch"], df["metrics/precision(B)"], label="precision")
plt.plot(df["epoch"], df["metrics/recall(B)"], label="recall")
plt.xlabel("epoch")
plt.ylabel("score")
plt.title("Validation precision and recall by epoch")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("precision_recall_by_epoch.png", dpi=200)
plt.show()

# box-loss and classification-loss
plt.figure(figsize=(7, 4))
plt.plot(df["epoch"], df["val/box_loss"], label="box-loss")
plt.plot(df["epoch"], df["val/cls_loss"], label="cls-loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("box loss and classification loss by epoch")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("loss_by_epoch.png", dpi=200)
plt.show()

# confusion matrix for the CNN
model.eval()
n = len(test_ds.classes)
cm = torch.zeros(n, n, dtype=torch.int64)
with torch.no_grad():
    for x, y in test_dl:
        pred = model(x.to(device)).argmax(1).cpu()
        cm += torch.bincount(pred * n + y, minlength=n * n).view(n, n)

cm_norm = cm / cm.sum(0, keepdim=True)

plt.figure(figsize=(6, 5))
plt.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.xticks(range(n), test_ds.classes, rotation=45)
plt.yticks(range(n), test_ds.classes)
plt.xlabel('True')
plt.ylabel('Predicted')
for i in range(n):
    for j in range(n):
        plt.text(j, i, f'{cm_norm[i, j]:.2f}\n({cm[i, j]})', ha='center', va='center',
                 color='white' if cm_norm[i, j] > 0.5 else 'black')
plt.title('CNN confusion matrix (test set)')
plt.colorbar()
plt.tight_layout()
plt.savefig('cnn_confusion_matrix.png', dpi=200)
plt.show()
